In [ ]:
# Setup & repo-root anchoring 
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path

sns.set_theme(style='whitegrid')

def _find_root(markers=('data', 'WESEffectsDiD')):
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if all((cand / m).exists() for m in markers):
            return cand
    raise FileNotFoundError(
        'Could not locate the repo root (expected data/ and WESEffectsDiD/). '
        f'Searched upward from {here}.')

ROOT = _find_root()
RESULTS = ROOT / 'ddd_results'
RESULTS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))
from result_table_style import save_result_table

SEX_TO_FEMALE = {'Women+': 1, 'Women': 1, 'Female': 1,
                 'Men+': 0,  'Men': 0,   'Male': 0}

def recode_sex(s):
    out = s.astype(str).str.strip().map(SEX_TO_FEMALE)
    if out.isna().any():
        bad = sorted(s[out.isna()].astype(str).str.strip().unique())
        raise ValueError(f'Unrecognised Sex label(s): {bad}')
    return out.astype(int)

POST_YEAR = 2018
print('ROOT =', ROOT)

In [ ]:
# Build the pooled Canada + Mexico frame (both genders, 2014-2024)
ca_se = pd.read_csv(ROOT / 'WESEffectsDiD' / 'data' / 'cleanedSelfEmployed.csv')
ca_lf = pd.read_csv(ROOT / 'WESEffectsDiD' / 'data' / 'cleanedLabourStats.csv')
ca = pd.merge(
    ca_se,
    ca_lf[['Province', 'Year', 'Sex', 'Control_LaborForce', 'Control_UnemploymentRate']],
    on=['Province', 'Year', 'Sex'], how='left')
ca = ca.rename(columns={'Province': 'geo', 'Year': 'year', 'Self_Employed': 'num',
                        'Control_LaborForce': 'lf', 'Control_UnemploymentRate': 'unemp'})
ca['female'] = recode_sex(ca['Sex'])
ca['country'] = 'Canada'; ca['canada'] = 1
ca_base = ca[['country', 'canada', 'geo', 'year', 'female', 'num', 'lf', 'unemp']].copy()

mx = pd.read_csv(ROOT / 'data' / 'mexico' / 'aggregatedData.csv')
mx_base = mx.rename(columns={'State': 'geo', 'Year': 'year', 'SelfEmployedPSTS': 'num',
                             'LaborForce': 'lf', 'UnemploymentRate': 'unemp'}).copy()
mx_base['female'] = recode_sex(mx_base['Sex'])
mx_base['country'] = 'Mexico'; mx_base['canada'] = 0
mx_base = mx_base[['country', 'canada', 'geo', 'year', 'female', 'num', 'lf', 'unemp']].copy()

pooled = pd.concat([ca_base, mx_base], ignore_index=True)
pooled['geo']  = pooled['country'] + ':' + pooled['geo'].astype(str)
pooled = pooled[pooled['year'].between(2014, 2024)].copy()
pooled['year'] = pooled['year'].astype(int)
pooled['rate']     = pooled['num'] / pooled['lf']
pooled['log_rate'] = np.log(pooled['rate'])
pooled['Post']   = (pooled['year'] >= POST_YEAR).astype(int)
pooled = pooled.dropna(subset=['log_rate', 'lf', 'unemp'])
pooled = pooled[np.isfinite(pooled['log_rate'])].copy()

print('Pooled rows:', len(pooled),
      '| clusters (geo):', pooled['geo'].nunique(),
      '| Canada:', int((pooled.country == 'Canada').sum()),
      '| Mexico:', int((pooled.country == 'Mexico').sum()),
      '| years:', sorted(pooled['year'].unique()))
pooled.head()

In [ ]:
# DDD plot
p = pooled.copy()
p['Gender'] = p['female'].map({0: 'Men', 1: 'Women'})
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.lineplot(ax=axes[0], data=p, x='year', y='log_rate', hue='country',
             style='Gender', markers=True, dashes=True, linewidth=2.2, errorbar=None)
axes[0].axvline(x=POST_YEAR, color='red', linestyle='--')
axes[0].set_title('Mean log(PSTS self-employed / labour force)')
axes[0].set_ylabel('Log rate'); axes[0].set_xticks(sorted(p['year'].unique()))
gap = (p.groupby(['country', 'year', 'female'])['log_rate'].mean().unstack('female'))
gap['gap'] = gap[1] - gap[0]; gap = gap.reset_index()
sns.lineplot(ax=axes[1], data=gap, x='year', y='gap', hue='country',
             marker='o', linewidth=2.6)
axes[1].axvline(x=POST_YEAR, color='red', linestyle='--', label='WES (2018)')
axes[1].set_title('Gender gap = log(women) - log(men)  [DDD object]')
axes[1].set_ylabel('Female - Male (log points)'); axes[1].set_xticks(sorted(p['year'].unique()))
axes[1].legend()
fig.suptitle('Canada vs Mexico — DDD, full period 2014-2024', fontsize=14)
fig.tight_layout()
fig.savefig(RESULTS / 'canadaMexico_ddd_fullperiod_allPSTS.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Triple-difference estimation (two-way FE, clustered by geo) 
DDD_FORMULA = ('log_rate ~ female + female:Post + female:canada + canada:Post '
               '+ female:canada:Post + unemp + C(geo) + C(year)')
res_ddd = smf.ols(DDD_FORMULA, data=pooled).fit(
    cov_type='cluster', cov_kwds={'groups': pooled['geo']})
print(res_ddd.summary())

triple = 'female:canada:Post'
print('\n=== DDD estimate of the WES effect (female:canada:Post) ===')
print('coef =', round(res_ddd.params[triple], 4),
      '| std err =', round(res_ddd.bse[triple], 4),
      '| p =', round(res_ddd.pvalues[triple], 4),
      '| 95% CI =', [round(x, 4) for x in res_ddd.conf_int().loc[triple].tolist()],
      '| n =', int(res_ddd.nobs))

save_result_table(
    'OLS Regression: Triple Difference (DDD), Canada vs. Mexico (2014–2024)',
    res_ddd.summary(), RESULTS / 'CanadaMexico_DDD_OLS_fullperiod_allPSTS.png',
    title_fontsize=13)
print('\nSaved results to', RESULTS)